# Data Exploration: Kaggle Retinal Fundus Dataset

This notebook explores the structure, distribution, and characteristics of the raw retinal fundus image dataset before preprocessing and model training. The dataset originates from the APTOS 2019 and Messidor-2 challenges, organized for diabetic retinopathy (DR) severity classification.

## Dataset Structure (M1 Raw)

The raw dataset (referred to as **M1**) is organized as follows:

```text
data/
├── Aptos_messidor_dataset/
│   ├── class_0/ (10384 images)
│   └── class_1/ (11526 images)
├── Test Images_new/
│   ├── class_0/ (1805 images)
│   └── class_1/ (1860 images)
│   └── data_aptos_test.csv
├── Train Images_new/
│   ├── class_0/ (7220 images)
│   └── class_1/ (7425 images)
│   └── data_aptos_train.csv
```   

This dataset is structured for binary classification: presence or absence of diabetic retinopathy. The original Kaggle competition featured a 5-class severity scale, but for this study we adopt a binary partition: **class_0 (No DR)** and **class_1 (any DR severity)**.

## Class Distribution and Data Splitting Strategy

The DR severity labels follow the International Clinical Diabetic Retinopathy scale:

- **0** → No DR
- **1** → Mild DR
- **2** → Moderate DR
- **3** → Severe DR
- **4** → Proliferative DR

For this super-resolution study, we consolidate all DR severity levels (1–4) into a single positive class, maintaining a **binary classification structure**. This allows us to later analyze whether super-resolution preprocessing improves diagnostic classification.

### Key Design Decision: On-the-Fly Degradation

All degradations (downsampling, blur, noise) will be applied **dynamically during training and evaluation** through a parameterized degradation pipeline. This approach:
- Eliminates the need to store 9 degraded versions of each image on disk
- Guarantees reproducibility through fixed random seeds
- Prevents data leakage between training and validation splits
- Enables systematic comparison across degradation scenarios

In [7]:
import os

def contar_archivos(ruta_carpeta):
    archivos = [
        f for f in os.listdir(ruta_carpeta)
        if os.path.isfile(os.path.join(ruta_carpeta, f))
    ]
    
    cantidad = len(archivos)
    print(f"Hay {cantidad} archivos en la carpeta '{ruta_carpeta}'")
    return cantidad

contar_archivos(r'data\raw\archive\Aptos_messidor_dataset\Aptos_messidor_dataset\class_0')
contar_archivos(r'data\raw\archive\Aptos_messidor_dataset\Aptos_messidor_dataset\class_1')
contar_archivos(r'data\raw\archive\Test Images_new\Test Images\class_0')
contar_archivos(r'data\raw\archive\Test Images_new\Test Images\class_1')
contar_archivos(r'data\raw\archive\Train Images_new\Train Images\class_0')
contar_archivos(r'data\raw\archive\Train Images_new\Train Images\class_1')

Hay 10384 archivos en la carpeta 'data\raw\archive\Aptos_messidor_dataset\Aptos_messidor_dataset\class_0'
Hay 11526 archivos en la carpeta 'data\raw\archive\Aptos_messidor_dataset\Aptos_messidor_dataset\class_1'
Hay 1805 archivos en la carpeta 'data\raw\archive\Test Images_new\Test Images\class_0'
Hay 1860 archivos en la carpeta 'data\raw\archive\Test Images_new\Test Images\class_1'
Hay 7220 archivos en la carpeta 'data\raw\archive\Train Images_new\Train Images\class_0'
Hay 7425 archivos en la carpeta 'data\raw\archive\Train Images_new\Train Images\class_1'


7425

There is:

- 21910 images (Aptos_messidor_dataset)

- 14645 images (Train Images_new)

- 3665 images (Test Images_new)

Total ≈ 40220 imágenes

We will keep a fixed train/validation/test split using only the high-resolution (HR) images, without creating separate degraded datasets on disk. All degradations (e.g., downsampling, blur, noise) will be applied dynamically during training and evaluation through a controlled and parameterized degradation function. This ensures reproducibility, avoids data leakage, and allows us to systematically study the impact of different degradation models while keeping the dataset organization clean and consistent.